# dxf_to_motorcad_region

# Explicit Notebook Cell Labels
- This notebook uses explicit labels `CELL-01`, `CELL-02`, ... inside markdown + code comments.
- Refer to these labels when discussing cells (independent of VS Code UI ordering).

**Run path**: `CELL-01` → `CELL-02` → `CELL-03` (optional `CELL-04`).

In [ ]:
# CELL-01: Set DXF path
dxfPath=r"E:\KDH\KJS\C688\Moasoft C588_CAD.dxf"

In [ ]:
# --- ezdxf addon(drawing) 기반: 레이어별로 골라서 그리기 (Motor-CAD 없이) ---
# 핵심 아이디어: draw_layout은 'OFF/FROZEN 레이어는 그리지 않음'을 이용합니다.
# 따라서 보고 싶은 레이어만 ON, 나머지는 OFF로 잠깐 바꾸고 draw_layout을 호출합니다.

import ezdxf
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
from ezdxf.addons.drawing import RenderContext, Frontend
from ezdxf.addons.drawing.matplotlib import MatplotlibBackend

def list_layers(dxf_path, top_n=30):
    doc = ezdxf.readfile(dxf_path)
    msp = doc.modelspace()
    layer_counter = Counter()
    type_by_layer = defaultdict(Counter)
    for e in msp:
        lname = getattr(e.dxf, 'layer', '(no layer)')
        layer_counter[lname] += 1
        type_by_layer[lname][e.dxftype()] += 1
    print('[Layers] entity counts by layer (top)')
    for name, cnt in layer_counter.most_common(top_n):
        types = ', '.join([f'{t}:{c}' for t, c in type_by_layer[name].most_common(6)])
        print(f'  {name:30s}  {cnt:6d}   {types}')
    return layer_counter

def draw_dxf_layers_addon(dxf_path, layers_to_show=None, figsize=(9, 9)):
    """
    ezdxf.addons.drawing으로 레이어를 선택해 렌더링합니다.
    - layers_to_show=None: 전체 레이어 표시(기본)
    - layers_to_show=[...]: 지정 레이어만 표시 (나머지는 임시로 OFF 처리 후 복구)
    """
    doc = ezdxf.readfile(dxf_path)
    msp = doc.modelspace()

    # 레이어 상태 백업
    layer_state = {}
    for layer in doc.layers:
        name = layer.dxf.name
        layer_state[name] = (getattr(layer, 'is_off', False), getattr(layer, 'is_frozen', False))

    try:
        if layers_to_show is not None and len(layers_to_show) > 0:
            keep = set(layers_to_show)
            # 선택 레이어만 ON, 나머지 OFF
            for layer in doc.layers:
                name = layer.dxf.name
                if name in keep:
                    try:
                        layer.on()
                        layer.thaw()
                    except Exception:
                        pass
                else:
                    try:
                        layer.off()
                    except Exception:
                        pass

        fig = plt.figure(figsize=figsize)
        ax = fig.add_axes([0, 0, 1, 1])
        ctx = RenderContext(doc)
        out = MatplotlibBackend(ax)
        Frontend(ctx, out).draw_layout(msp, finalize=True)
        ax.set_aspect('equal', adjustable='box')
        plt.show()
        return fig, ax
    finally:
        # 레이어 상태 복구(안전)
        for layer in doc.layers:
            name = layer.dxf.name
            if name not in layer_state:
                continue
            was_off, was_frozen = layer_state[name]
            try:
                if was_off:
                    layer.off()
                else:
                    layer.on()
                if was_frozen:
                    layer.freeze()
                else:
                    layer.thaw()
            except Exception:
                pass


In [ ]:

# 1) 레이어 목록 출력
layers= list_layers(dxfPath)


In [ ]:
# --- DXF '원점' 후보 찾기 (ezdxf only) ---
# DXF에서 말하는 '원점'은 상황에 따라 다릅니다:
# 1) WCS (0,0)
# 2) $INSBASE (삽입 기준점)
# 3) UCS 원점($UCSORG / $PUCSORG)
# 4) 도면 extents 중심(EXTMIN/EXTMAX)
# 여기서는 후보들을 전부 출력하고, (0,0) 위치도 시각적으로 확인합니다.

import ezdxf
import math
import matplotlib.pyplot as plt
from ezdxf.addons.drawing import RenderContext, Frontend
from ezdxf.addons.drawing.matplotlib import MatplotlibBackend

doc = ezdxf.readfile(dxfPath)
msp = doc.modelspace()

def _get_header_point(name):
    v = doc.header.get(name, None)
    if v is None:
        return None
    try:
        # ezdxf는 (x,y,z) tuple 또는 Vec3 류로 들어올 수 있음
        return (float(v[0]), float(v[1]), float(v[2]) if len(v) > 2 else 0.0)
    except Exception:
        try:
            return (float(v.x), float(v.y), float(getattr(v, 'z', 0.0)))
        except Exception:
            return None

insbase = _get_header_point('$INSBASE')
ucsorg = _get_header_point('$UCSORG')
pucsorg = _get_header_point('$PUCSORG')
extmin = _get_header_point('$EXTMIN')
extmax = _get_header_point('$EXTMAX')

print('[DXF Origin Candidates]')
print('  WCS origin: (0, 0, 0)')
print('  $INSBASE  :', insbase)
print('  $UCSORG   :', ucsorg)
print('  $PUCSORG  :', pucsorg)
print('  $EXTMIN   :', extmin)
print('  $EXTMAX   :', extmax)
if extmin and extmax:
    ext_center = ((extmin[0] + extmax[0]) / 2.0, (extmin[1] + extmax[1]) / 2.0, (extmin[2] + extmax[2]) / 2.0)
    print('  extents center:', ext_center)

# 엔티티 좌표 분포 기반: 실제 geometry bbox
xmin = ymin = float('inf')
xmax = ymax = float('-inf')
n_sample = 0

def _acc(x, y):
    global xmin, ymin, xmax, ymax, n_sample
    xmin = min(xmin, x)
    ymin = min(ymin, y)
    xmax = max(xmax, x)
    ymax = max(ymax, y)
    n_sample += 1

for e in msp:
    t = e.dxftype()
    if t == 'LINE':
        _acc(float(e.dxf.start.x), float(e.dxf.start.y))
        _acc(float(e.dxf.end.x), float(e.dxf.end.y))
    elif t == 'CIRCLE':
        cx, cy = float(e.dxf.center.x), float(e.dxf.center.y)
        r = float(e.dxf.radius)
        _acc(cx - r, cy - r)
        _acc(cx + r, cy + r)
    elif t == 'ARC':
        # start/end만 샘플링(빠르게)
        cx, cy = float(e.dxf.center.x), float(e.dxf.center.y)
        r = float(e.dxf.radius)
        sa = math.radians(float(e.dxf.start_angle))
        ea = math.radians(float(e.dxf.end_angle))
        _acc(cx + r * math.cos(sa), cy + r * math.sin(sa))
        _acc(cx + r * math.cos(ea), cy + r * math.sin(ea))
    elif t == 'LWPOLYLINE':
        pts = e.get_points(format='xy')
        for (x, y) in pts:
            _acc(float(x), float(y))

if n_sample > 0:
    bbox_center = ((xmin + xmax) / 2.0, (ymin + ymax) / 2.0)
    print('  geometry bbox: ', (xmin, ymin), ' ~ ', (xmax, ymax))
    print('  geometry bbox center:', bbox_center)
else:
    print('  geometry bbox: (no samples)')

# 시각화: addon 렌더 + (0,0) 축 / 후보 포인트 표시
fig = plt.figure(figsize=(9, 9))
ax = fig.add_axes([0, 0, 1, 1])
ctx = RenderContext(doc)
out = MatplotlibBackend(ax)
Frontend(ctx, out).draw_layout(msp, finalize=True)

# WCS origin crosshair
ax.axhline(0, color='red', linewidth=0.8)
ax.axvline(0, color='red', linewidth=0.8)
ax.plot([0], [0], 'ro', markersize=4, label='(0,0)')

def _plot_pt(pt, label, color):
    if pt is None:
        return
    ax.plot([pt[0]], [pt[1]], marker='x', color=color, markersize=8)
    ax.text(pt[0], pt[1], f' {label}', color=color, fontsize=9)

_plot_pt(insbase, 'INSBASE', 'orange')
_plot_pt(ucsorg, 'UCSORG', 'purple')
_plot_pt(pucsorg, 'PUCSORG', 'green')
if extmin and extmax:
    _plot_pt(ext_center, 'EXT center', 'blue')
if n_sample > 0:
    _plot_pt((bbox_center[0], bbox_center[1], 0.0), 'Geom bbox center', 'black')

ax.set_aspect('equal', adjustable='box')
ax.grid(True, linewidth=0.3)
ax.legend(loc='best')
plt.show()

In [ ]:

# 2) 레이어 선택해서 표시
layers_to_show = None  # 예: ['0', 'SomeLayerName']
draw_dxf_layers_addon(dxfPath, layers_to_show=layers_to_show, figsize=(9, 9))

ezdxf

In [ ]:
import matplotlib.pyplot as plt
from ezdxf.addons.drawing import RenderContext, Frontend
from ezdxf.addons.drawing.matplotlib import MatplotlibBackend

doc = ezdxf.readfile(dxfPath)
msp = doc.modelspace()
# Create a figure
fig = plt.figure()
ax = fig.add_axes([0, 0, 1, 1])
ctx = RenderContext(doc)
out = MatplotlibBackend(ax)
Frontend(ctx, out).draw_layout(msp, finalize=True)

# Show the plot
plt.show()

In [ ]:
import ezdxf
import math
from ansys.motorcad.core.geometry import Coordinate, Region, Line, Arc, rt_to_xy
from ansys.motorcad.core.geometry_drawing import draw_objects

# ------------------------------------------------------------
# 공통 유틸 (노트북 전역에서 재사용)
# - 셀 실행 순서 의존을 줄이기 위해 여기 한 곳에 모읍니다.
# - 아래에서 정의되는 import 함수들은 이 헬퍼들에만 의존하도록 유지합니다.
# ------------------------------------------------------------

def _snap_xy(x, y, tol):
    if tol is None or tol <= 0:
        return (float(x), float(y))
    tol = float(tol)
    return (round(float(x) / tol) * tol, round(float(y) / tol) * tol)

def is_same_point(p1, p2, tol=1e-5):
    return abs(p1.x - p2.x) < tol and abs(p1.y - p2.y) < tol

def bulge_to_arc(start_point, end_point, bulge):
    """Convert bulge information to Motor-CAD Arc parameters. Bulge = tan(theta/4)."""
    if bulge == 0:
        return Line(start_point, end_point)
    theta = 4 * math.atan(bulge)
    dist = abs(end_point - start_point)
    if dist == 0:
        return None
    radius = dist / (2 * math.sin(abs(theta) / 2))
    # Motor-CAD Arc radius sign convention: + for CCW(bulge>0), - for CW(bulge<0)
    mc_radius = radius if bulge > 0 else -radius
    return Arc(start_point, end_point, radius=mc_radius)

def ensure_ccw(region):
    """Ensure region entities are CCW when possible."""
    if region.is_closed():
        try:
            if not region.entities.is_anticlockwise:
                region.entities.reverse()
        except Exception as e:
            print(f"Warning during CCW check: {e}")

## --- DXF -> Region (stitch 방식) : 끝점 스냅 + 스티칭 ---

In [ ]:
import copy
from collections import Counter

def _snap_key(coord, tol):
    if tol is None or tol <= 0:
        return (coord.x, coord.y)
    return (round(coord.x / tol) * tol, round(coord.y / tol) * tol)

def _canonicalize_endpoints(entities, tol):
    """
    모든 엔티티의 start/end를 tolerance 기반 격자에 스냅하고,
    같은 점은 같은 Coordinate 객체를 공유하도록 canonicalize 합니다.
    """
    canonical = {}
    def canon(c):
        kx, ky = _snap_key(c, tol)
        key = (kx, ky)
        if key not in canonical:
            canonical[key] = Coordinate(kx, ky)
        return canonical[key]
    for ent in entities:
        ent.start = canon(ent.start)
        ent.end = canon(ent.end)
    return entities

def stitch_entities_snapped(entities, tolerance=1e-5, report=True):
    """끝점을 스냅 후 greedy 스티칭으로 폐곡선을 찾습니다 (접점 degree>=3이면 누락 가능)."""
    if not entities:
        return []
    pool = entities[:]
    _canonicalize_endpoints(pool, tolerance)

    regions = []
    dropped_chains = 0
    dropped_entities = 0

    while pool:
        current_chain = Region()
        current_chain.add_entity(pool.pop(0))
        chain_closed = False

        while True:
            last_pt = current_chain.entities[-1].end
            first_pt = current_chain.entities[0].start
            if last_pt.x == first_pt.x and last_pt.y == first_pt.y:
                current_chain.entities[-1].end = first_pt
                chain_closed = True
                break

            found_next = False
            for i, cand in enumerate(pool):
                if cand.start.x == last_pt.x and cand.start.y == last_pt.y:
                    cand.start = last_pt
                    current_chain.add_entity(pool.pop(i))
                    found_next = True
                    break
                if cand.end.x == last_pt.x and cand.end.y == last_pt.y:
                    cand.reverse()
                    cand.start = last_pt
                    current_chain.add_entity(pool.pop(i))
                    found_next = True
                    break

            if not found_next:
                break

        if chain_closed:
            ensure_ccw(current_chain)
            regions.append(current_chain)
        else:
            dropped_chains += 1
            dropped_entities += len(current_chain.entities)

    if report and dropped_chains > 0:
        print(f"[stitch] WARNING: {dropped_chains} chain(s) not closed; {dropped_entities} entity(ies) dropped.")
        print("  -> 접점(degree>=3) 또는 열린 윤곽(open contour)이면 greedy 스티칭이 실패할 수 있습니다")
        print(f"  -> tolerance(grid snap) = {tolerance}")

    return regions

def import_dxf_as_regions_stitch(dxf_path, tolerance=0.1, report=True):
    """LINE/LWPOLYLINE/CIRCLE/ARC를 읽고 스티칭으로 Region 리스트 생성."""
    doc = ezdxf.readfile(dxf_path)
    msp = doc.modelspace()

    regions = []
    loose_entities = []

    for entity in msp:
        dxftype = entity.dxftype()

        if dxftype == 'LINE':
            start = Coordinate(entity.dxf.start.x, entity.dxf.start.y)
            end = Coordinate(entity.dxf.end.x, entity.dxf.end.y)
            loose_entities.append(Line(start, end))

        elif dxftype == 'LWPOLYLINE':
            # bulge를 안전하게 읽기 위해 x,y,bulge 포맷 사용
            points = entity.get_points(format='xyb')
            poly_entities = []
            for i in range(len(points) - 1):
                (x1, y1, b1) = points[i]
                (x2, y2, _b2) = points[i + 1]
                start = Coordinate(x1, y1)
                end = Coordinate(x2, y2)
                if b1 == 0:
                    poly_entities.append(Line(start, end))
                else:
                    arc = bulge_to_arc(start, end, b1)
                    if arc:
                        poly_entities.append(arc)

            if entity.closed and points:
                (x1, y1, b1) = points[-1]
                (x2, y2, _b2) = points[0]
                start = Coordinate(x1, y1)
                end = Coordinate(x2, y2)
                if b1 == 0:
                    poly_entities.append(Line(start, end))
                else:
                    arc = bulge_to_arc(start, end, b1)
                    if arc:
                        poly_entities.append(arc)

                _canonicalize_endpoints(poly_entities, tolerance)
                reg = Region()
                for e in poly_entities:
                    reg.add_entity(e)
                ensure_ccw(reg)
                regions.append(reg)
            else:
                loose_entities.extend(poly_entities)

        elif dxftype == 'CIRCLE':
            center = Coordinate(entity.dxf.center.x, entity.dxf.center.y)
            radius = entity.dxf.radius
            p1 = center + Coordinate(radius, 0)
            p2 = center - Coordinate(radius, 0)

            reg = Region()
            reg.add_entity(Arc(p1, p2, center, radius))
            reg.add_entity(Arc(p2, p1, center, radius))
            ensure_ccw(reg)
            regions.append(reg)

        elif dxftype == 'ARC':
            center = Coordinate(entity.dxf.center.x, entity.dxf.center.y)
            radius = entity.dxf.radius
            sx, sy = rt_to_xy(radius, entity.dxf.start_angle)
            ex, ey = rt_to_xy(radius, entity.dxf.end_angle)
            start = center + Coordinate(sx, sy)
            end = center + Coordinate(ex, ey)
            loose_entities.append(Arc(start, end, center, radius))

    if loose_entities:
        stitched = stitch_entities_snapped(loose_entities, tolerance, report=report)
        regions.extend(stitched)

    if report:
        print(f"[import_dxf_as_regions_stitch] closed regions: {len(regions)}")
    return regions

In [ ]:
# --- Robust fallback: Shapely polygonize (ARC는 선분 근사) ---

In [ ]:
# 목적: 접점(degree>=3)에서도 폐곡선(외곽/내부 루프)을 안정적으로 생성
# 주의: ARC를 Motor-CAD Arc로 복원하지 않고 Line으로 반환합니다 (정밀도/속도는 파라미터로 조절).

import math
import time
from collections import Counter
from shapely.geometry import LineString, MultiLineString
from shapely.ops import unary_union, polygonize

def _dxf_arc_angles_ccw(start_deg, end_deg):
    """DXF ARC는 start->end로 CCW 진행. end<start면 360을 더해 wrap 처리."""
    s = float(start_deg)
    e = float(end_deg)
    if e < s:
        e += 360.0
    return s, e

def _step_deg_from_max_error(radius, max_error):
    """
    원호를 선분으로 근사할 때 chord error(최대 처짐) <= max_error가 되도록 step 각도를 계산합니다.
    error = r * (1 - cos(theta/2))  =>  theta = 2*acos(1 - error/r)
    """
    r = float(radius)
    e = float(max_error)
    if r <= 0 or e <= 0:
        return None
    if e >= r:
        # 너무 큰 error는 180deg까지 허용해도 충분
        return 180.0
    v = 1.0 - (e / r)
    # 수치 안정성
    v = min(1.0, max(-1.0, v))
    theta = 2.0 * math.degrees(math.acos(v))
    # theta가 너무 작으면 과도 분할이므로 하한을 둠
    return max(theta, 0.1)

def _arc_to_linestring(
    center,
    radius,
    start_angle_deg,
    end_angle_deg,
    *,
    step_deg=2.0,
    max_error=None,
    max_points=2000,
    snap_tol=0.1,
    min_step_deg=0.1,
    max_step_deg=30.0,
    ):
    """
    ARC를 LineString으로 변환합니다.
    - step_deg: 고정 스텝(도)
    - max_error: chord error 기반 자동 스텝(우선)
    - max_points: 원호당 최대 점 개수 제한(폭주 방지)
    """
    s, e = _dxf_arc_angles_ccw(start_angle_deg, end_angle_deg)
    total = e - s
    if total <= 0:
        return None

    use_step = float(step_deg)
    if max_error is not None:
        auto = _step_deg_from_max_error(radius, max_error)
        if auto is not None:
            use_step = auto
    use_step = min(max_step_deg, max(min_step_deg, use_step))

    # 최소 2점, 최대 max_points
    n = int(math.ceil(total / use_step)) + 1
    n = max(2, min(int(max_points), n))

    pts = []
    for k in range(n):
        ang = math.radians(s + total * (k / (n - 1)))
        x = center.x + float(radius) * math.cos(ang)
        y = center.y + float(radius) * math.sin(ang)
        pts.append(_snap_xy(x, y, snap_tol))

    cleaned = [pts[0]]
    for p in pts[1:]:
        if p != cleaned[-1]:
            cleaned.append(p)
    if len(cleaned) < 2:
        return None
    return LineString(cleaned)

def import_dxf_as_regions_polygonize(
    dxf_path,
    tolerance=0.1,
    arc_step_deg=2.0,
    arc_max_error=None,
    arc_max_points=2000,
    do_unary_union=True,
    report=True,
    ):
    """
    LINE/ARC를 polygonize로 폐곡선 Region 리스트로 변환.

    성능 팁(대형 DXF에서 중요):
    - `arc_step_deg`를 키우면(예: 2→5) 빠르지만 곡선이 덜 부드러워집니다.
    - `arc_max_error`를 쓰면(예: tolerance/2) 필요한 만큼만 분할되어 보통 더 빠릅니다.
    - `arc_max_points`로 원호당 분할 폭주를 방지합니다.
    - `do_unary_union=False`는 더 빠를 수 있지만 교차/접점 분할이 부족해 루프가 누락될 수 있습니다.
    """
    t0 = time.perf_counter()
    doc = ezdxf.readfile(dxf_path)
    msp = doc.modelspace()

    lines = []
    handled = 0
    skipped = Counter()
    arc_points_total = 0
    line_count = 0
    arc_count = 0

    for e in msp:
        t = e.dxftype()
        if t == "LINE":
            p1 = _snap_xy(e.dxf.start.x, e.dxf.start.y, tolerance)
            p2 = _snap_xy(e.dxf.end.x, e.dxf.end.y, tolerance)
            if p1 != p2:
                lines.append(LineString([p1, p2]))
                handled += 1
                line_count += 1
        elif t == "ARC":
            center = e.dxf.center
            r = float(e.dxf.radius)
            ls = _arc_to_linestring(
                center,
                r,
                e.dxf.start_angle,
                e.dxf.end_angle,
                step_deg=arc_step_deg,
                max_error=arc_max_error,
                max_points=arc_max_points,
                snap_tol=tolerance,
            )
            if ls is not None:
                lines.append(ls)
                handled += 1
                arc_count += 1
                # shapely LineString은 coords로 길이 확인 가능
                arc_points_total += len(ls.coords)
        else:
            skipped[t] += 1

    t1 = time.perf_counter()
    if report:
        print(f"[polygonize] handled LINE/ARC: {handled}, skipped others: {sum(skipped.values())}")
        print(f"[polygonize] lines: {line_count}, arcs: {arc_count}, arc points total: {arc_points_total}")
        if skipped:
            print("[polygonize] skipped types:")
            for k, v in skipped.most_common():
                print(f"  {k:12s}: {v}")
        print(f"[polygonize] build lines time: {t1 - t0:.3f}s")

    if not lines:
        return []

    # GEOS 쪽 연산(대부분 시간이 여기서 소요)
    ml = MultiLineString([ls for ls in lines])
    if do_unary_union:
        t2 = time.perf_counter()
        merged = unary_union(ml)
        t3 = time.perf_counter()
        if report:
            print(f"[polygonize] unary_union time: {t3 - t2:.3f}s")
    else:
        merged = ml

    t4 = time.perf_counter()
    polys = list(polygonize(merged))
    t5 = time.perf_counter()
    if report:
        print(f"[polygonize] polygonize time: {t5 - t4:.3f}s")
        print(f"[polygonize] polygons found: {len(polys)}")

    regions = []
    def ring_to_region(coords):
        reg = Region()
        for i in range(len(coords) - 1):
            x1, y1 = coords[i]
            x2, y2 = coords[i + 1]
            reg.add_entity(Line(Coordinate(x1, y1), Coordinate(x2, y2)))
        ensure_ccw(reg)
        return reg

    for poly in polys:
        regions.append(ring_to_region(list(poly.exterior.coords)))
        for interior in poly.interiors:
            regions.append(ring_to_region(list(interior.coords)))

    if report:
        t6 = time.perf_counter()
        print(f"[polygonize] regions returned (exteriors + holes): {len(regions)}")
        print(f"[polygonize] TOTAL time: {t6 - t0:.3f}s")
    return regions

## 뭐지

In [ ]:
from collections import Counter, defaultdict
import ezdxf

# --- DXF 진단: 어떤 엔티티 타입이 들어있는지 확인 ---
print("[DXF Diagnose] path:", dxfPath)
doc = ezdxf.readfile(dxfPath)
msp = doc.modelspace()

print("[DXF Diagnose] dxfversion:", getattr(doc, "dxfversion", "(unknown)"))
insunits = doc.header.get("$INSUNITS", None)
print("[DXF Diagnose] $INSUNITS:", insunits, "(0이면 unitless인 경우가 많습니다)")

# 1) Modelspace 엔티티 타입 분포
type_counter = Counter()
layer_counter = Counter()
unsupported = Counter()
handled = {"LINE", "LWPOLYLINE", "CIRCLE", "ARC"}
insert_count = 0

for e in msp:
    t = e.dxftype()
    type_counter[t] += 1
    layer_name = getattr(e.dxf, "layer", "(no layer)")
    layer_counter[layer_name] += 1
    if t == "INSERT":
        insert_count += 1
    if t not in handled:
        unsupported[t] += 1

print("\n[DXF Diagnose] --- Modelspace entity type counts ---")
for t, c in type_counter.most_common():
    print(f"  {t:12s} : {c}")

print("\n[DXF Diagnose] --- Types NOT handled by import_dxf_as_regions ---")
if unsupported:
    for t, c in unsupported.most_common():
        print(f"  {t:12s} : {c}")
else:
    print("  (none)")

# 2) INSERT(블록 참조) 안에 실제로 어떤 엔티티가 있는지(virtual_entities)
print("\n[DXF Diagnose] INSERT count:", insert_count)
if insert_count > 0:
    ve_counter = Counter()
    ve_layer_counter = Counter()
    failed_inserts = 0
    for e in msp.query("INSERT"):
        try:
            for ve in e.virtual_entities():
                ve_counter[ve.dxftype()] += 1
                ve_layer_counter[getattr(ve.dxf, "layer", "(no layer)")] += 1
        except Exception:
            failed_inserts += 1
    print("[DXF Diagnose] --- INSERT virtual entity type counts ---")
    for t, c in ve_counter.most_common():
        print(f"  {t:12s} : {c}")
    if failed_inserts:
        print("[DXF Diagnose] warning: virtual_entities() failed for", failed_inserts, "INSERT(s)")

# 3) 레이어 상태(OFF/FROZEN)와 엔티티 분포
print("\n[DXF Diagnose] --- Layer status (top by entity count) ---")
layer_table = []
for layer_name, count in layer_counter.most_common():
    try:
        layer = doc.layers.get(layer_name)
        is_off = bool(getattr(layer, "is_off", False))
        is_frozen = bool(getattr(layer, "is_frozen", False))
        is_locked = bool(getattr(layer, "is_locked", False))
    except Exception:
        is_off = is_frozen = is_locked = False
    layer_table.append((layer_name, count, is_off, is_frozen, is_locked))

for layer_name, count, is_off, is_frozen, is_locked in layer_table[:30]:
    flags = []
    if is_off: flags.append("OFF")
    if is_frozen: flags.append("FROZEN")
    if is_locked: flags.append("LOCKED")
    flag_str = (",".join(flags) if flags else "-")
    print(f"  {layer_name:24s} : {count:6d}   [{flag_str}]")

problem_layers = [(n,c,off,frz) for (n,c,off,frz,_) in layer_table if c > 0 and (off or frz)]
if problem_layers:
    print("\n[DXF Diagnose] NOTE: OFF/FROZEN 레이어에 엔티티가 있습니다. (렌더링/변환에서 누락처럼 보일 수 있음)")
    for n,c,off,frz in problem_layers[:20]:
        print(f"  {n:24s} : {c:6d}   [OFF={off}, FROZEN={frz}]")

# 4) 빠질 가능성이 높은 타입 요약
likely_missing = [t for t in ("INSERT","SPLINE","ELLIPSE","POLYLINE","HATCH") if type_counter.get(t, 0) > 0]
print("\n[DXF Diagnose] likely-missing types present:", likely_missing if likely_missing else "(none)")

In [ ]:
import math
from collections import defaultdict, Counter

# --- 진단 2: 끝점 연결 그래프(degree) 분석 ---
# LINE/ARC endpoints degree를 확인합니다.
doc = ezdxf.readfile(dxfPath)
msp = doc.modelspace()

def _arc_endpoints(e):
    center = e.dxf.center
    r = e.dxf.radius
    sa = e.dxf.start_angle
    ea = e.dxf.end_angle
    sx, sy = rt_to_xy(r, sa)
    ex, ey = rt_to_xy(r, ea)
    return (center.x + sx, center.y + sy), (center.x + ex, center.y + ey)

tol = 0.1  # import에서 쓰는 tolerance와 동일하게 맞추는 것을 권장
deg = Counter()
by_point = defaultdict(list)
ent_total = 0

for e in msp:
    t = e.dxftype()
    if t == "LINE":
        (x1, y1) = (e.dxf.start.x, e.dxf.start.y)
        (x2, y2) = (e.dxf.end.x, e.dxf.end.y)
    elif t == "ARC":
        (x1, y1), (x2, y2) = _arc_endpoints(e)
    else:
        continue

    p1 = _snap_xy(x1, y1, tol)
    p2 = _snap_xy(x2, y2, tol)
    deg[p1] += 1
    deg[p2] += 1
    by_point[p1].append(t)
    by_point[p2].append(t)
    ent_total += 1

print("[Endpoint Graph] tolerance =", tol)
print("[Endpoint Graph] entity count (LINE/ARC) =", ent_total)
degree_hist = Counter(deg.values())
print("[Endpoint Graph] degree histogram:")
for k in sorted(degree_hist):
    print(f"  degree {k}: {degree_hist[k]}")

dangling = [p for p, d in deg.items() if d == 1]
junctions = [p for p, d in deg.items() if d >= 3]
print("\n[Endpoint Graph] dangling endpoints (degree=1):", len(dangling))
print("[Endpoint Graph] junction endpoints (degree>=3):", len(junctions))

if dangling:
    print("\n[Endpoint Graph] 샘플 dangling endpoints (최대 20개):")
    for p in dangling[:20]:
        print(" ", p, "types:", Counter(by_point[p]))

if junctions:
    print("\n[Endpoint Graph] 샘플 junction endpoints (최대 20개):")
    for p in junctions[:20]:
        print(" ", p, "degree:", deg[p], "types:", Counter(by_point[p]))

## 실행 

In [ ]:
# 실행
use_polygonize = True  # 접점(degree>=3) 때문에 누락이 있으면 True 권장
if use_polygonize:
    # 성능 튜닝 포인트:
    # - arc_max_error: tolerance보다 조금 작게(예: tolerance/2) 주면 대체로 빠르고 품질도 충분합니다.
    # - arc_step_deg: max_error를 안 쓸 때만 의미가 큽니다(값을 키우면 빨라짐).
    tol = 0.03
    regions = import_dxf_as_regions_polygonize(
        dxfPath,
        tolerance=tol,
        arc_step_deg=1,
        arc_max_error=tol / 2,
        arc_max_points=2800,
        do_unary_union=True,
        report=True,
    )
else:
    regions = import_dxf_as_regions_stitch(dxfPath, tolerance=0.03, report=True)

print(f"Found {len(regions)} closed regions.")

In [ ]:
# regions 리스트 전체를 하나의 drawing_region에 합쳐서 한 번에 draw
if 'regions' not in locals() or regions is None:
    raise RuntimeError("먼저 실행 셀에서 regions를 생성하세요.")

drawing_region = Region()
for reg in regions:
    if reg is None or not getattr(reg, 'entities', None):
        continue
    for ent in reg.entities:
        drawing_region.add_entity(ent)

draw_objects(drawing_region)

In [ ]:
import ansys.motorcad.core as pymotorcad
from ansys.motorcad.core.geometry import Arc, Coordinate, Line, rt_to_xy, xy_to_rt
from ansys.motorcad.core.geometry_drawing import draw_objects_debug

# Connect to Motor-CAD
mc = pymotorcad.MotorCAD(open_new_instance=False)

# Reset geometry to default
mc.reset_adaptive_geometry()

In [ ]:

def get_barrier_centre_and_radius(coordinate_1, coordinate_2, coordinate_3, arc_direction):
    """Calculate barrier arc centre and radius coordinates.

    Parameters
    ----------
    coordinate_1 : Coordinate
        Arc start coordinate.

    coordinate_2 : Coordinate
        Extra coordinate on arc

    coordinate_3 : Coordinate
        Arc end coordinate

    arc_direction : Integer
        Direction to create arc between start/end

    Returns
    -------
    radius : float
        Arc radius

    centre : Coordinate
        Arc centre coordinate
    """
    _, start_t = xy_to_rt(coordinate_1.x, coordinate_1.y)
    _, end_t = xy_to_rt(coordinate_3.x, coordinate_3.y)

    a = (
        coordinate_1.x * (coordinate_2.y - coordinate_3.y)
        - coordinate_1.y * (coordinate_2.x - coordinate_3.x)
        + coordinate_2.x * coordinate_3.y
        - coordinate_3.x * coordinate_2.y
    )
    b = (
        (coordinate_1.x**2 + coordinate_1.y**2) * (coordinate_3.y - coordinate_2.y)
        + (coordinate_2.x**2 + coordinate_2.y**2) * (coordinate_1.y - coordinate_3.y)
        + (coordinate_3.x**2 + coordinate_3.y**2) * (coordinate_2.y - coordinate_1.y)
    )
    c = (
        (coordinate_1.x**2 + coordinate_1.y**2) * (coordinate_2.x - coordinate_3.x)
        + (coordinate_2.x**2 + coordinate_2.y**2) * (coordinate_3.x - coordinate_1.x)
        + (coordinate_3.x**2 + coordinate_3.y**2) * (coordinate_1.x - coordinate_2.x)
    )
    centre = Coordinate((-b / a) / 2, (-c / a) / 2)
    radius = Line(centre, coordinate_1).length * arc_direction

    return radius, centre


def get_pockets_include_corner_rounding():
    """Whether corner rounding should be applied to pocket.

    Returns
    -------
        boolean
    """
    return (mc.get_variable("CornerRounding_Rotor") == 1) and (
        mc.get_variable("CornerRoundingRadius_Rotor") > 0
    )


def get_rotor_mirror_line():
    """Create mirror line through rotor from origin to airgap.

    Returns
    -------
        Line
    """
    rotor_radius = mc.get_variable("RotorDiameter")
    number_poles = mc.get_variable("Pole_Number")
    airgap_centre_x, airgap_centre_y = rt_to_xy(rotor_radius, (360 / number_poles) / 2)

    return Line(Coordinate(0, 0), Coordinate(airgap_centre_x, airgap_centre_y))


def get_coordinates(pocket, coordinate_indices, mirror_line=None):
    """Get list of coordinates from pocket using coordinate_indices.
    Coordinates ordered : start, end, extra coordinate on arc.

    Parameters
    ----------
    pocket : Region
        Pocket region

    coordinate_indices : list of integer
        Pocket region coordinate indices

    mirror_line : Line
        Mirror line to generate extra coordinate on arc

    Returns
    -------
        list of Coordinate
    """
    coordinates = [pocket.points[index] for index in coordinate_indices]
    if mirror_line is not None:
        coordinates.append(coordinates[0].mirror(mirror_line))

    return coordinates


def get_coordinates_no_centre_post(pocket):
    """Get list of coordinates to use to generate top and bottom arcs for pocket.
    Coordinates ordered : start, end, extra coordinate for each arc.

    Returns
    -------
        list of Coordinate
    """
    if get_pockets_include_corner_rounding():
        return get_coordinates(pocket, [2, 8, 5, 11, 17, 14])
    else:
        return get_coordinates(pocket, [1, 5, 3, 6, 0, 8])


def get_coordinates_centre_post(pocket):
    """Get list of coordinates to use to generate top and bottom arcs for pocket.
    Coordinates ordered : start, end, extra coordinate for each arc.

    Parameters
    ----------
    pocket : Region
        Pocket region

    Returns
    -------
        list of Coordinate
    """
    mirror = get_rotor_mirror_line()
    if get_pockets_include_corner_rounding():
        return get_coordinates(pocket, [2, 5], mirror) + get_coordinates(
            pocket, [8, 11], mirror_line=mirror
        )
    else:
        return get_coordinates(pocket, [1, 3], mirror) + get_coordinates(
            pocket, [4, 0], mirror_line=mirror
        )


def update_pocket_geometry(pocket, coordinates):
    """Update the pocket entities to create curved pocket using input coordinates.

    Parameters
    ----------
    pocket : Region
        Pocket region

    coordinates : list of Coordinate
        Coordinates to generate arcs with in region
    """
    entities = []
    # create list of arc entities from coordinates
    for element in range(0, len(coordinates), 3):
        if (element + 1) % 2 == 0:
            # clockwise arc
            arc_direction = -1
        else:
            # ant-clockwise arc
            arc_direction = 1

        # get arc radius and centre point from three coordinates across the arc
        radius, centre = get_barrier_centre_and_radius(
            coordinates[element],  # arc start coordinate
            coordinates[element + 1],  # arc end coordinate
            coordinates[element + 2],  # arc third coordinate
            arc_direction,
        )
        entities.append(
            Arc(
                coordinates[element],
                coordinates[element + 1],
                centre,
                radius,
            )
        )
    # remove entities between start and end coordinates of each arc then insert into pocket
    for count, element in enumerate(range(0, len(coordinates), 3)):
        start_index = pocket.points.index(coordinates[element])
        end_index = pocket.points.index(coordinates[element + 1])
        if end_index == 0:
            end_index = len(pocket.points)

        # use index of start/end coordinates to remove pockets entities between them
        for index in reversed(range(start_index, end_index)):
            pocket.entities.pop(index)

        pocket.insert_entity(start_index, entities[count])


def get_pocket_name(index):
    """Return unique pocket name used in Motor-CAD.

    Parameters
    ----------
    index : integer
        Current pocket index

    Returns
    -------
        string
    """
    if index == 0:
        return "Rotor Pocket"
    else:
        return "Rotor Pocket_" + str(index)



### Adaptive Template 3

In [ ]:
# Reset geometry to default
mc.reset_adaptive_geometry()

# Use the ``get_region()`` method to get the required regions 
# and store these in the ``standard_regions`` list.
MagnetRegions = [
    mc. get_region("L1_1Magnet1"),
    mc. get_region("L1_1Magnet2"),
    mc. get_region("L2_1Magnet1"),
    mc. get_region("L2_1Magnet2"),
    # mc. get_region("RotorDuctFluidRegion_1"),
    # mc. get_region("RotorDuctFluidRegion_2")
]
replacement_Magnet_regions = [
    mc.get_region_dxf("DXFRegion_Rotor_7"),
    mc.get_region_dxf("DXFRegion_Rotor_2"),
    mc.get_region_dxf("DXFRegion_Rotor_16"),
    mc.get_region_dxf("DXFRegion_Rotor_11"),

]
# PocketRegions = [
#     mc. get_region("Rotor Pocket_1"),
#     mc. get_region("Rotor Pocket_2"),
#     mc. get_region("Rotor Pocket_4"),
#     mc. get_region("Rotor Pocket"),
#     # mc. get_region("RotorDuctFluidRegion_1"),
#     # mc. get_region("RotorDuctFluidRegion_2")
# ]
# replacement_regions = [
#     mc.get_region_dxf("DXFRegion_Rotor_6"),
#     mc.get_region_dxf("DXFRegion_Rotor_7"),
#     mc.get_region_dxf("DXFRegion_Rotor_10"),
#     mc.get_region_dxf("DXFRegion_Rotor_11"),
#     mc.get_region_dxf("DXFRegion_Rotor_14"),
#     mc.get_region_dxf("DXFRegion_Rotor_12")
# ]

for index in range(len(MagnetRegions)): 
    i = MagnetRegions[index]
    i.replace(replacement_Magnet_regions[index])
    mc.set_region(i)         

In [ ]:
# --- DXF에서 만든 regions를 Motor-CAD Adaptive Geometry로 적용하기 ---
# 목표: 지금 노트북에서 만든 `regions`를 Motor-CAD에 set_region()으로 주입해서
#       Adaptive Templates(스크립트)에서 쓰는 'Region 객체'로 활용

import copy
import ansys.motorcad.core as pymotorcad
from ansys.motorcad.core.geometry import Region, RegionType, EntityList
from shapely.geometry import Polygon

def _region_area_linear(region):
    """현재 region이 Line들로만 구성되어 있을 때 면적을 계산(분류용)."""
    if not region.entities:
        return 0.0
    coords = [(region.entities[0].start.x, region.entities[0].start.y)]
    for ent in region.entities:
        coords.append((ent.end.x, ent.end.y))
    poly = Polygon(coords)
    if not poly.is_valid:
        poly = poly.buffer(0)
    return float(poly.area)

def _clone_entities(src_region):
    # Motor-CAD에 보낼 때는 독립 객체가 안전합니다
    return EntityList([copy.deepcopy(e) for e in src_region.entities])

def apply_dxf_regions_to_motorcad(
    regions,
    parent_region_name="Rotor",
    replace_parent_geometry=True,
    pocket_region_type=RegionType.rotor_air,
    pocket_material="Air",
    pocket_colour=(255, 255, 255),
    open_new_instance=False,
):
    """
    DXF에서 인식한 regions를 Motor-CAD에 적용합니다.
    - 가장 면적이 큰 region을 '외곽(부모 region 형상)'으로 간주
    - 나머지는 '포켓/홀(자식 region)'로 간주하여 parent 아래에 추가

    주의:
    - `parent_region_name`은 Motor-CAD 모델에 이미 존재해야 합니다 (예: Rotor/Stator).
    - Motor-CAD가 실행 중이 아니면 open_new_instance=True로 새로 열 수 있습니다.
    """
    if not regions:
        raise ValueError("regions가 비어있습니다. 먼저 DXF import 셀을 실행하세요.")

    # 1) 외곽/내부 분리(면적 기준)
    areas = [(_region_area_linear(r), i) for i, r in enumerate(regions)]
    areas.sort(reverse=True)
    outer_idx = areas[0][1]
    outer = regions[outer_idx]
    pockets = [r for i, r in enumerate(regions) if i != outer_idx]

    print(f"[Apply] outer index={outer_idx}, pockets={len(pockets)}")

    # 2) Motor-CAD 연결
    mc = pymotorcad.MotorCAD(open_new_instance=open_new_instance)
    mc.set_variable("MessageDisplayState", 2)  # 팝업 최소화

    # Adaptive Templates 모드로 쓰려면 Motor-CAD UI에서도 Geometry Template Type=Adaptive여야 합니다
    # (외부 실행이라도, 적용 후 스크립트로 저장/로드 가능)

    # 3) 부모 region 가져오고(예: Rotor) 형상 교체
    parent = mc.get_region(parent_region_name)
    if replace_parent_geometry:
        parent.entities = _clone_entities(outer)
        # region_type 경고 방지용(필수는 아님):
        parent._region_type = RegionType.adaptive
        if not parent.is_closed():
            print("[Apply] WARNING: outer region이 closed가 아닙니다. DXF 변환을 확인하세요.")
        mc.set_region(parent)
        print(f"[Apply] Updated parent region geometry: {parent_region_name}")

    # 4) 포켓/홀을 자식 region으로 추가
    #    기존에 같은 이름이 있으면 충돌할 수 있으니 prefix를 고정합니다.
    for n, src in enumerate(pockets, start=1):
        pocket = Region(region_type=pocket_region_type, motorcad_instance=mc)
        pocket.name = f"DXF_Pocket_{n}"
        pocket.parent = parent
        pocket.material = pocket_material
        pocket.colour = pocket_colour
        pocket.duplications = parent.duplications
        pocket.entities = _clone_entities(src)
        pocket._region_type = pocket_region_type
        if not pocket.is_closed():
            print(f"[Apply] WARNING: pocket {n} not closed")
            continue
        mc.set_region(pocket)

    print("[Apply] Done. Motor-CAD Geometry -> Editor에서 regions를 확인하세요.")
    return mc

# 실행 예시
# - Motor-CAD를 이미 켜둔 상태면 open_new_instance=False 권장
# - Rotor가 아니라 Stator에 넣고 싶으면 parent_region_name을 바꾸세요

try:
    mc = apply_dxf_regions_to_motorcad(
        regions,
        parent_region_name="Rotor",
        replace_parent_geometry=True,
        open_new_instance=False,
    )
except Exception as e:
    print("[Apply] 실패:", e)
    print("- Motor-CAD를 실행한 뒤 다시 시도하거나")
    print("- open_new_instance=True로 새 인스턴스를 열어보세요")
    print("- parent_region_name(예: 'Rotor')가 현재 모델에 존재하는지도 확인하세요")

In [ ]:
from tools.motorCAD.pyMCAD import dxf_to_entitylists, dxf_entitylists_to_regions
ent_map = dxf_to_entitylists(dxfPath, line_tolerance=1e-4, arc_tolerance=1e-4)
# regions = dxf_entitylists_to_regions(ent_map, name_prefix=\"DXF\")

In [ ]:
regions = dxf_entitylists_to_regions(ent_map)

In [ ]:
regions

## sharply

In [ ]:
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
from ansys.motorcad.core.geometry import Region, Line, Arc, Coordinate
from ansys.motorcad.core.geometry_drawing import draw_objects

# --- Helper Functions ---

def region_to_shapely(region, discretization_step=5.0):
    """
    Convert a Motor-CAD Region object to a Shapely Polygon.
    Arcs are discretized into line segments.
    """
    points = []
    if not region.entities:
        return None

    # Start with the start point of the first entity
    first_entity = region.entities[0]
    points.append((first_entity.start.x, first_entity.start.y))
    
    for entity in region.entities:
        if isinstance(entity, Line):
            points.append((entity.end.x, entity.end.y))
        elif isinstance(entity, Arc):
            # Discretize arc
            total_angle = entity.total_angle
            num_segments = int(np.ceil(total_angle / discretization_step))
            if num_segments < 1: num_segments = 1
                
            for i in range(1, num_segments + 1):
                fraction = i / num_segments
                coord = entity.get_coordinate_from_distance(entity.start, fraction=fraction)
                points.append((coord.x, coord.y))
                
    poly = Polygon(points)
    if not poly.is_valid:
        poly = poly.buffer(0)
    return poly

def shapely_to_regions(shapely_geom):
    """
    Convert a Shapely geometry (Polygon or MultiPolygon) back to a list of Motor-CAD Region objects.
    Note: Curves are converted to linear segments (Lines).
    """
    regions = []
    
    if shapely_geom.is_empty:
        return regions

    if isinstance(shapely_geom, Polygon):
        geoms = [shapely_geom]
    elif isinstance(shapely_geom, MultiPolygon):
        geoms = shapely_geom.geoms
    else:
        return regions

    for geom in geoms:
        reg = Region()
        # Exterior ring
        coords = list(geom.exterior.coords)
        # coords includes the closing point (same as start), so iterate up to len-1
        for i in range(len(coords) - 1):
            start = Coordinate(coords[i][0], coords[i][1])
            end = Coordinate(coords[i+1][0], coords[i+1][1])
            reg.add_entity(Line(start, end))
        regions.append(reg)
        
    return regions

# --- Operation Functions ---

def subtract_regions_python(region1, region2):
    p1 = region_to_shapely(region1)
    p2 = region_to_shapely(region2)
    
    if p1 is None or p2 is None:
        return [region1] # Or handle error
        
    result_poly = p1.difference(p2)
    return shapely_to_regions(result_poly)

def unite_regions_python(region1, region2):
    p1 = region_to_shapely(region1)
    p2 = region_to_shapely(region2)
    
    if p1 is None: return region2
    if p2 is None: return region1
    
    result_poly = unary_union([p1, p2])
    # Unite usually returns a single region if they overlap, or multiple if disjoint
    return shapely_to_regions(result_poly)

def check_collision_python(region1, region2):
    p1 = region_to_shapely(region1)
    p2 = region_to_shapely(region2)
    
    if p1 is None or p2 is None:
        return False
        
    return p1.intersects(p2) and not p1.touches(p2)

# --- Test ---

# Create two overlapping squares
r1 = Region()
r1.add_entity(Line(Coordinate(0,0), Coordinate(10,0)))
r1.add_entity(Line(Coordinate(10,0), Coordinate(10,10)))
r1.add_entity(Line(Coordinate(10,10), Coordinate(0,10)))
r1.add_entity(Line(Coordinate(0,10), Coordinate(0,0)))
r1.name = "Square 1"

r2 = Region()
r2.add_entity(Line(Coordinate(5,5), Coordinate(15,5)))
r2.add_entity(Line(Coordinate(15,5), Coordinate(15,15)))
r2.add_entity(Line(Coordinate(15,15), Coordinate(5,15)))
r2.add_entity(Line(Coordinate(5,15), Coordinate(5,5)))
r2.name = "Square 2"

print(f"Collision Check: {check_collision_python(r1, r2)}")

# Subtract r2 from r1
result_regions = subtract_regions_python(r1, r2)

print(f"Resulting regions after subtraction: {len(result_regions)}")

# Visualize
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

# Plot Original
ax[0].set_title("Original")
for entity in r1.entities:
    ax[0].plot([entity.start.x, entity.end.x], [entity.start.y, entity.end.y], 'b-')
for entity in r2.entities:
    ax[0].plot([entity.start.x, entity.end.x], [entity.start.y, entity.end.y], 'r-')

# Plot Subtraction Result
ax[1].set_title("Subtraction (Blue - Red)")
for reg in result_regions:
    for entity in reg.entities:
        ax[1].plot([entity.start.x, entity.end.x], [entity.start.y, entity.end.y], 'g-', linewidth=2)
        
# Plot Union Result
union_regions = unite_regions_python(r1, r2)
ax[2].set_title("Union")
for reg in union_regions:
    for entity in reg.entities:
        ax[2].plot([entity.start.x, entity.end.x], [entity.start.y, entity.end.y], 'k-', linewidth=2)

for a in ax: a.set_aspect('equal')
plt.show()

In [ ]:
# --- Workflow Example: Import DXF -> Clean Geometry -> Ready for Analysis ---

# 1. Import from DXF (Assumed 'regions' is loaded from previous cells)
# regions = import_dxf_as_regions(dxfPath)

print("Processing geometry for analysis...")

# 2. Convert to Shapely for Cleaning
# This step is crucial. It fixes self-intersections, closes open loops, 
# and merges overlapping entities which often cause errors in analysis SW.

shapely_polys = []
# 'regions' variable is expected to be available from the previous cell execution
if 'regions' in locals():
    for reg in regions:
        poly = region_to_shapely(reg)
        if poly and not poly.is_empty:
            shapely_polys.append(poly)
else:
    print("Variable 'regions' not found. Please run the previous cell.")

if not shapely_polys:
    print("Error: No valid geometry found.")
else:
    # 3. Clean Topology
    # unary_union is effective at resolving complex topology issues
    # It merges overlapping polygons and handles self-intersections
    cleaned_poly = unary_union(shapely_polys)
    
    # 4. Convert back to Motor-CAD Regions
    # The result is a list of clean, closed regions ready for export/analysis
    cleaned_regions = shapely_to_regions(cleaned_poly)
    print(f"Geometry cleaned. Resulting independent regions: {len(cleaned_regions)}")
    
    # 5. Visualize Cleaned Geometry
    fig, ax = plt.subplots(figsize=(8, 8))
    
    for i, reg in enumerate(cleaned_regions):
        # Plot each region with a different color to verify separation
        x_vals = []
        y_vals = []
        for entity in reg.entities:
            x_vals.append(entity.start.x)
            y_vals.append(entity.start.y)
        # Close the loop for plotting
        if x_vals:
            x_vals.append(reg.entities[-1].end.x)
            y_vals.append(reg.entities[-1].end.y)
            
        ax.fill(x_vals, y_vals, alpha=0.5, label=f'Region {i+1}')
        ax.plot(x_vals, y_vals, 'k-', linewidth=1)
        
    ax.set_aspect('equal')
    ax.set_title("Cleaned Geometry (Ready for Analysis)")
    ax.legend()
    ax.grid(True)
    plt.show()
    
    # 6. (Conceptual) Export to Motor-CAD or File
    # for reg in cleaned_regions:
    #     mc.set_region(reg)  # Send to Motor-CAD
    #     # or
    #     # export_to_dxf(reg, "cleaned_model.dxf")

# dxf_to_entitylists

In [ ]:
from tools.motorCAD.pyMCAD import dxf_to_entitylists, dxf_entitylists_to_regions
ent_by_layer = dxf_to_entitylists(dxfPath)
regions= dxf_entitylists_to_regions(ent_by_layer)

In [ ]:
# CELL-02: DXF → Motor-CAD Regions (stitch=True)
import importlib

# Reload to pick up newly-added re-exports (dxf_to_regions, etc.)
import tools.motorCAD.pyMCAD as pyMCAD
import tools.motorCAD.pyMCAD.melec_req_check as mreq
importlib.reload(mreq)
importlib.reload(pyMCAD)

from tools.motorCAD.pyMCAD import dxf_to_regions
from ansys.motorcad.core.geometry import RegionType

# DXF를 스티칭해서 폐곡선 루프를 Region으로 변환
stitch_tol = 1e-4
regions = dxf_to_regions(
    dxfPath,
    stitch=True,
    stitch_tol=stitch_tol,
    stitch_by_layer=True,
 )

## CELL-02: DXF → Motor-CAD Regions (stitch=True)
Creates `regions` using endpoint stitching (fast, no widgets).

In [28]:
# Debug: inspect stitched IR size
import importlib
import tools.motorCAD.pyMCAD.dxf_geometry as dg
importlib.reload(dg)

ir = dg.dxf_to_ir(dxfPath, stitch=True, stitch_tol=stitch_tol, stitch_by_layer=True)
print("n_primitives:", len(ir["primitives"]))
print("n_loops:", len(ir["loops"]))
print("meta keys:", list(ir.get("meta", {}).keys()))
if ir["loops"]:
    print("first loop n_edges:", len(ir["loops"][0].primitive_indices), "layer:", ir["loops"][0].layer)

n_primitives: 175
n_loops: 0
meta keys: ['dxf_path', 'layers', 'n_primitives', 'n_loops']


In [29]:
# Quick sweep: see how stitch_tol affects loop detection
import numpy as np
tols = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
for tol in tols:
    ir_tmp = dg.dxf_to_ir(dxfPath, stitch=True, stitch_tol=tol, stitch_by_layer=True)
    print(f"tol={tol:>7g}  primitives={len(ir_tmp['primitives'])}  loops={len(ir_tmp['loops'])}")

tol=  1e-06  primitives=175  loops=0
tol=  1e-05  primitives=175  loops=0
tol= 0.0001  primitives=175  loops=0
tol=  0.001  primitives=175  loops=0
tol=   0.01  primitives=175  loops=0
tol=    0.1  primitives=175  loops=0
tol=      1  primitives=175  loops=0


In [30]:
# Debug: endpoint graph stats at a given tolerance
tol = 0.1
prims = dg.primitives_from_dxf(dxfPath)
node_ids = {}
next_id = 0
def _nid(pt):
    global next_id
    k = dg._pt_key(pt[0], pt[1], tol)
    if k not in node_ids:
        node_ids[k] = len(node_ids)
    return node_ids[k]
degrees = {}
for i,p in enumerate(prims):
    u = _nid(p.start)
    v = _nid(p.end)
    degrees[u] = degrees.get(u,0) + 1
    degrees[v] = degrees.get(v,0) + 1
import collections
deg_counts = collections.Counter(degrees.values())
print("nodes:", len(degrees), "edges:", len(prims))
print("degree counts:", dict(sorted(deg_counts.items())))
print("degree1 nodes:", sum(1 for d in degrees.values() if d==1))
print("degree2 nodes:", sum(1 for d in degrees.values() if d==2))
print("degree>2 nodes:", sum(1 for d in degrees.values() if d>2))

nodes: 160 edges: 175
degree counts: {2: 132, 3: 26, 4: 2}
degree1 nodes: 0
degree2 nodes: 132
degree>2 nodes: 28


In [31]:
# Try stitching without layer separation (some DXFs put each segment on a unique layer)
ir_all = dg.dxf_to_ir(dxfPath, stitch=True, stitch_tol=0.1, stitch_by_layer=False)
print("stitch_by_layer=False => loops:", len(ir_all["loops"]))

stitch_by_layer=False => loops: 8


In [ ]:
from collections import Counter

print("regions type:", type(regions))
print("regions length:", len(regions))
print("region item type:", type(regions[0]) if regions else None)

# RegionType distribution
def _region_type_name(r):
    rt = getattr(r, "region_type", None)
    # rt might be RegionType enum or a raw string depending on construction
    return getattr(rt, "name", None) or getattr(rt, "value", None) or str(rt)

type_counts = Counter(_region_type_name(r) for r in regions)
print("\nRegionType counts:")
for k, v in type_counts.most_common():
    print(f"  {k}: {v}")

# Name sanity checks
names = [getattr(r, "name", "") for r in regions]
print("\nUnique names:", len(set(names)), "/", len(names))
print("First 10 names:", names[:10])

In [ ]:
# CELL-03: Validate area/centroid (fast check, no widgets)
from tools.motorCAD.pyMCAD import regions_summary_df

df_regions = regions_summary_df(regions, sort_by="area", ascending=False)
display(df_regions)

# Quick stats
closed_count = int((df_regions["is_closed"] == True).sum())
area_nonzero = int((df_regions["area"].fillna(0) > 0).sum())
print(f"closed (is_closed=True): {closed_count} / {len(df_regions)}")
print(f"area > 0: {area_nonzero} / {len(df_regions)}")
print("top 10 by area:")
display(df_regions[["idx","name","region_type","is_closed","area","n_entities"]].head(10))

,idx,name,region_type,is_closed,area,centroid_x,centroid_y,n_entities
0,0,DXF_0_loop_0,dxf_import,True,0.0,0,0,6
1,1,DXF_0_loop_1,dxf_import,True,0.0,0,0,8
2,2,DXF_0_loop_2,dxf_import,True,0.0,0,0,8
3,3,DXF_0_loop_3,dxf_import,True,0.0,0,0,8
4,4,DXF_0_loop_4,dxf_import,True,0.0,0,0,8
5,5,DXF_0_loop_5,dxf_import,True,0.0,0,0,8
6,6,DXF_0_loop_6,dxf_import,True,0.0,0,0,8
7,7,DXF_0_loop_7,dxf_import,True,0.0,0,0,5


Output()

## CELL-03: Validate area/centroid (fast)
Shows `regions` summary and key counts (closed, area>0).

In [ ]:
# CELL-04 (optional): Draw top-N regions (avoid widget slowness)
from ansys.motorcad.core.geometry_drawing import draw_objects
from tools.motorCAD.pyMCAD import regions_summary_df

N = 20
df = regions_summary_df(regions, sort_by="area", ascending=False)
idxs = df["idx"].head(N).tolist()
to_draw = [regions[i] for i in idxs]
print(f"Drawing top {len(to_draw)} regions by area...")
draw_objects(to_draw, label_regions=False, full_geometry=False, legend=None, title=f"Top {len(to_draw)} regions by area")

## CELL-04 (optional): Draw top-N regions (no widgets)
Draws only the largest regions to keep rendering fast.

In [ ]:
# (disabled) Pattern/exact mapping experiments
## 필요할 때만 아래 로직을 다시 활성화해서 사용하세요.
# - layer_region_type_patterns / layer_region_type_map 기반 매핑 실험 블록

In [ ]:
# (disabled) Full draw of all regions (can be slow)
# from ansys.motorcad.core.geometry_drawing import draw_objects
# draw_objects(regions)

In [ ]:
# (disabled) group_strategy='layer' quick checks (kept for reference)
# dxf_to_entitylists(dxfPath, group_strategy="layer")
# dxf_to_regions(dxfPath, group_strategy="layer")

In [ ]:
# (disabled) empty cell

In [ ]:
# (disabled) Duplicate summary/area checks (Cell 40 covers this now)

In [ ]:
# (disabled) Deep diagnostics: ent_by_layer / entities inspection

In [ ]:
# (disabled) Deep diagnostics: first region entity dump

In [ ]:
# (disabled) DXF raw analysis by layer/entity-type (ezdxf)

In [ ]:
# (disabled) Long explanatory print block (kept in history)

In [ ]:
# (disabled) Reload-new-DXF experiment block

In [ ]:
# (disabled) DXF raw deep analysis sample listing (ezdxf)

In [ ]:
# (disabled) Step 1: new DXF re-check experiment

In [ ]:
# (disabled) Step 2: geometry_fitting module exploration

In [ ]:
# (disabled) Step 3: geometry_fitting.return_entity_list() stitching experiment

In [ ]:
# (disabled) Step 4: endpoint-graph stitching + return_entity_list experiment